In [ ]:
import pandas as pd
import numpy as np

# ---------------------------
# Inputs (edit paths)
# ---------------------------
# Your peak calls (from the earlier notebook) – must have: chr, mid, ID, strand, score
tss_peaks_csv = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/prediction/results_intergenic_TSS/predictions_all.csv"      # TSS per-site scores
tts_peaks_csv = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/prediction/results_intergenic_TTS/predictions_all_TTS_thr0.7.csv"  # TTS per-site scores
# If you saved only peaks, use those. If you saved per-bin, you can keep the highest per local region.

# PlasmoDB annotation (GFF or GTF). We’ll parse minimal fields.
gff_path = "/rhome/zli529/lab/PlasmoDB_Genome/PlasmoDB_v48/PlasmoDB-48_Pfalciparum3D7.gff"

out_dir = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/prediction/"

In [ ]:
# ---------------------------
# Tunable windows (bp)
# ---------------------------
# Attach TSS peaks to known genes if the peak is within these windows of the annotated start by strand
PROM_UP = 500     # allowed upstream of annotated start
PROM_DOWN = 200   # allowed downstream of annotated start

# Attach TTS peaks if within these windows of annotated end by strand
TERM_UP = 200     # allowed upstream of annotated end
TERM_DOWN = 500   # allowed downstream of annotated end

# Novel unit pairing limits (TSS -> TTS on same strand)
NOVEL_MIN_LEN = 200     # min distance TSS->TTS
NOVEL_MAX_LEN = 4000    # max distance TSS->TTS
GENE_BUFFER   = 500     # how far from any known gene to still count as "intergenic/novel"

In [ ]:
# ---------------------------
# 1) Load peaks
# ---------------------------
def load_pred(path):
    df = pd.read_csv(path)
    # If you have lots of per-bin rows, you can thin to local maxima first.
    # Assuming already local-peaks or you’ll filter by score later:
    df = df.copy()
    df["pos"] = df["mid"].astype(int)
    df["strand"] = df["strand"].astype(str)
    return df[["chr","pos","ID","strand","score"]]

tss = load_pred(tss_peaks_csv).rename(columns={"pos":"tss_pos", "score":"tss_score", "ID":"tss_id"})
tts = load_pred(tts_peaks_csv).rename(columns={"pos":"tts_pos", "score":"tts_score", "ID":"tts_id"})


In [ ]:
# ---------------------------
# 2) Minimal GFF parse -> gene table
# ---------------------------
gcols = ['seqid','source','type','start','end','score','strand','phase','attributes']
gff = pd.read_csv(gff_path, sep="\t", comment="#", header=None, names=gcols)
# Keep genes / mRNA features (use whichever marks transcript bounds in your file)
genes = gff[gff["type"].isin(["gene","mRNA","transcript"])].copy()

# Extract gene_id/name from attributes
def extract_attr(attr, keys=("ID","Name","gene_id","gene_name","Parent")):
    for k in keys:
        m = pd.Series(attr).str.extract(fr"{k}=([^;]+)")
        if m.notna().any().iloc[0]:
            return m[0]
    return pd.Series([np.nan]*len(attr))

genes["gene_id"] = extract_attr(genes["attributes"])
genes = genes.dropna(subset=["gene_id"])

genes["start"] = genes["start"].astype(int)
genes["end"]   = genes["end"].astype(int)
genes = genes.rename(columns={"seqid":"chr"})
genes = genes[["chr","start","end","strand","gene_id","type"]].copy()

# Collapse to one entry per gene_id (take min start, max end across mRNA/exons if present)
gene_bounds = (genes
               .groupby(["chr","strand","gene_id"], as_index=False)
               .agg(start=("start","min"), end=("end","max")))

# Compute annotated TSS/TTS per strand
gene_bounds["ann_tss"] = np.where(gene_bounds["strand"]=="+", gene_bounds["start"], gene_bounds["end"])
gene_bounds["ann_tts"] = np.where(gene_bounds["strand"]=="+", gene_bounds["end"],   gene_bounds["start"])


In [ ]:
# ---------------------------
# 3) Attach TSS peaks to known genes (by strand & window)
# ---------------------------
tss_att = tss.merge(gene_bounds, on=["chr","strand"], how="inner")

# distance to annotated start (strand-aware)
tss_att["dist_to_ann_tss"] = np.where(
    tss_att["strand"]=="+", tss_att["tss_pos"] - tss_att["ann_tss"],  # +: peak - start
    tss_att["ann_tss"] - tss_att["tss_pos"]                           # -: start - peak (peak must be upstream on -)
)

# keep peaks within promoter window [-PROM_UP, +PROM_DOWN]
keep_tss = tss_att[(tss_att["dist_to_ann_tss"] >= -PROM_UP) & (tss_att["dist_to_ann_tss"] <= PROM_DOWN)].copy()

# best TSS per gene: highest score (or closest; choose your policy)
pc_tss = (keep_tss.sort_values(["gene_id","tss_score"], ascending=[True,False])
                  .drop_duplicates("gene_id"))
pc_tss = pc_tss[["chr","strand","gene_id","ann_tss","tss_pos","tss_score","dist_to_ann_tss"]]


In [ ]:
# ---------------------------
# 4) Attach TTS peaks to known genes (by strand & window)
# ---------------------------
tts_att = tts.merge(gene_bounds, on=["chr","strand"], how="inner")

tts_att["dist_to_ann_tts"] = np.where(
    tts_att["strand"]=="+", tts_att["tts_pos"] - tts_att["ann_tts"],  # +: peak - end (should be >= -TERM_UP & <= TERM_DOWN)
    tts_att["ann_tts"] - tts_att["tts_pos"]                           # -: end - peak
)

keep_tts = tts_att[(tts_att["dist_to_ann_tts"] >= -TERM_UP) & (tts_att["dist_to_ann_tts"] <= TERM_DOWN)].copy()

pc_tts = (keep_tts.sort_values(["gene_id","tts_score"], ascending=[True,False])
                 .drop_duplicates("gene_id"))
pc_tts = pc_tts[["chr","strand","gene_id","ann_tts","tts_pos","tts_score","dist_to_ann_tts"]]


In [ ]:
# ---------------------------
# 5) Save protein-coding attachments
# ---------------------------
pc_tss.to_csv(f"{out_dir}/protein_coding_TSS.tsv", sep="\t", index=False)
pc_tts.to_csv(f"{out_dir}/protein_coding_TTS.tsv", sep="\t", index=False)

In [ ]:
# ===== Normalize 'chr' and 'strand' columns everywhere =====
def normalize_chr_strand(df, who="df"):
    # try common alternatives
    chr_candidates = ["chr", "chrom", "chromosome", "seqid", "seqname", "scaffold"]
    strand_candidates = ["strand", "Strand"]

    chr_col = next((c for c in chr_candidates if c in df.columns), None)
    if chr_col is None:
        raise KeyError(f"[{who}] missing chromosome column; saw: {list(df.columns)[:10]}")
    if chr_col != "chr":
        df = df.rename(columns={chr_col: "chr"})

    strand_col = next((c for c in strand_candidates if c in df.columns), None)
    if strand_col is None:
        # some peak tables may not carry strand; if so, fill with '.'
        df = df.assign(strand=".")
    elif strand_col != "strand":
        df = df.rename(columns={strand_col: "strand"})

    # coerce dtypes
    df["chr"] = df["chr"].astype(str)
    df["strand"] = df["strand"].astype(str)
    return df

# Apply to all inputs used below
tss = normalize_chr_strand(tss, "tss")
tts = normalize_chr_strand(tts, "tts")
genes = normalize_chr_strand(genes, "genes")

# Sanity prints
print("tss cols:", list(tss.columns)[:10])
print("tts cols:", list(tts.columns)[:10])
print("genes cols:", list(genes.columns)[:10])

# Confirm required coordinate columns exist
if "tss_pos" not in tss.columns:
    # If your TSS table has 'pos' instead, rename it:
    if "pos" in tss.columns: tss = tss.rename(columns={"pos": "tss_pos"})
    else: raise KeyError("tss is missing 'tss_pos' (or 'pos') column.")

if "tts_pos" not in tts.columns:
    # If your TTS table has 'pos' instead, rename it:
    if "pos" in tts.columns: tts = tts.rename(columns={"pos": "tts_pos"})
    else: raise KeyError("tts is missing 'tts_pos' (or 'pos') column.")

# Also ensure gene bounds are present
for c in ["start","end"]:
    if c not in genes.columns:
        raise KeyError(f"genes missing '{c}' column; you may be passing the wrong table.")

# (Re)build quick per-chrom gene table used below
genes_chr_groups = {k: g[["start","end"]].sort_values(["start","end"]).reset_index(drop=True)
                    for k, g in genes.groupby("chr")}


In [ ]:
print(tss)
print(tts)

In [ ]:
import numpy as np
import pandas as pd

# ===== 0) Normalize columns and sanity checks =====
def norm_chr_strand(df, who):
    chr_alias = [c for c in ["chr","chrom","chromosome","seqid","seqname","scaffold"] if c in df.columns]
    if not chr_alias: raise KeyError(f"[{who}] missing chr column; saw {list(df.columns)[:12]}")
    if chr_alias[0] != "chr": df = df.rename(columns={chr_alias[0]:"chr"})

    if "strand" not in df.columns:
        df["strand"] = "."
    df["chr"] = df["chr"].astype(str)
    df["strand"] = df["strand"].astype(str)
    return df

# Expect your TSS/TTS frames as variables `tss` and `tts`.
tss = norm_chr_strand(tss, "tss")
tts = norm_chrand(tts, "tts") if 'norm_chrand' in globals() else norm_chr_strand(tts, "tts")  # typo-safety

# Map coordinate/score columns
if "tss_pos" not in tss.columns:
    if "pos" in tss.columns: tss = tss.rename(columns={"pos":"tss_pos"})
    elif "mid" in tss.columns: tss = tss.rename(columns={"mid":"tss_pos"})
    else: raise KeyError("tss missing tss_pos/pos/mid")
if "tss_score" not in tss.columns:
    if "score" in tss.columns: tss = tss.rename(columns={"score":"tss_score"})
    else: tss["tss_score"] = np.nan

if "tts_pos" not in tts.columns:
    if "pos" in tts.columns: tts = tts.rename(columns={"pos":"tts_pos"})
    elif "mid" in tts.columns: tts = tts.rename(columns={"mid":"tts_pos"})
    else: raise KeyError("tts missing tts_pos/pos/mid")
if "tts_score" not in tts.columns:
    if "score" in tts.columns: tts = tts.rename(columns={"score":"tts_score"})
    else: tts["tts_score"] = np.nan

# Drop obvious NA rows
tss = tss.dropna(subset=["chr","strand","tss_pos"]).copy()
tts = tts.dropna(subset=["chr","strand","tts_pos"]).copy()
tss["tss_pos"] = tss["tss_pos"].astype(int)
tts["tts_pos"] = tts["tts_pos"].astype(int)

print("TSS rows:", len(tss), "TTS rows:", len(tts))
print("TSS chr/strand examples:", tss[["chr","strand"]].drop_duplicates().head().to_dict("records"))
print("TTS chr/strand examples:", tts[["chr","strand"]].drop_duplicates().head().to_dict("records"))

# ===== 1) Confirm you actually used PEAKS (not all bins). If not, thin by local maxima+score. =====
def local_maxima_1d(a, k=3):
    n = len(a)
    if n == 0: return np.zeros(0, dtype=bool)
    m = np.ones(n, dtype=bool)
    for off in range(1, k+1):
        left  = np.r_[a[off:], np.full(off, -np.inf)]
        right = np.r_[np.full(off, -np.inf), a[:-off]]
        m &= (a > left) & (a > right)
    return m

MIN_PROB_FOR_PEAK_TSS = 0.6   # try 0.6 if too strict
MIN_PROB_FOR_PEAK_TTS = 0.6

def keep_local_peaks(df, pos_col, score_col, k=3, pmin=0.7):
    out = []
    for (chrom, strand), grp in df.groupby(["chr","strand"], sort=False):
        a = grp[score_col].to_numpy()
        mask = local_maxima_1d(a, k)
        sub = grp.loc[mask & (a >= pmin)].copy()
        out.append(sub)
    return pd.concat(out, ignore_index=True) if out else df.iloc[0:0].copy()

if "tss_is_peak" not in tss.columns:  # avoid double-thinning
    tss_peaks = keep_local_peaks(tss, "tss_pos", "tss_score", k=3, pmin=MIN_PROB_FOR_PEAK_TSS)
else:
    tss_peaks = tss.copy()

if "tts_is_peak" not in tts.columns:
    tts_peaks = keep_local_peaks(tts, "tts_pos", "tts_score", k=3, pmin=MIN_PROB_FOR_PEAK_TTS)
else:
    tts_peaks = tts.copy()

print("After local-peak filter: TSS:", len(tss_peaks), "TTS:", len(tts_peaks))

# ===== 2) Filter TSS to be intergenic (away from any annotated gene) =====
# Expect `genes` with columns: chr,start,end,strand (we only need start/end here)
genes = norm_chr_strand(genes, "genes")
for c in ["start","end"]:
    if c not in genes.columns: raise KeyError(f"genes missing {c}")
genes["start"] = genes["start"].astype(int)
genes["end"]   = genes["end"].astype(int)

GENE_BUFFER = 500  # try 200–1000
genes_by_chr = {k: g[["start","end"]].sort_values(["start","end"]).to_numpy()
                for k,g in genes.groupby("chr")}

def min_dist_to_any_interval(points, intervals):
    if intervals is None or len(intervals)==0:
        return np.full(len(points), np.inf, dtype=np.float64)
    s = intervals[:,0]; e = intervals[:,1]
    out = np.empty(len(points), dtype=np.float64)
    for i, p in enumerate(points):
        inside = (p >= s) & (p <= e)
        if inside.any():
            out[i] = 0.0
        else:
            out[i] = np.min(np.minimum(np.abs(p - s), np.abs(p - e)))
    return out

novel_tss_parts = []
for chrom, grp in tss_peaks.groupby("chr"):
    intervals = genes_by_chr.get(chrom, None)
    d = min_dist_to_any_interval(grp["tss_pos"].to_numpy(), intervals)
    keep = grp.copy()
    keep["dist_to_gene"] = d
    keep = keep[keep["dist_to_gene"] >= GENE_BUFFER]
    novel_tss_parts.append(keep)
novel_tss = pd.concat(novel_tss_parts, ignore_index=True) if novel_tss_parts else tss_peaks.iloc[0:0].copy()
print("Novel TSS after gene-distance filter:", len(novel_tss))

# ===== 3) Pair each novel TSS to nearest downstream TTS on the same strand =====
NOVEL_MIN_LEN = 200   # try 200
NOVEL_MAX_LEN = 4000  # try 6000

pairs = []
for (chrom, strand), grp in novel_tss.groupby(["chr","strand"], sort=False):
    tts_grp = tts_peaks[(tts_peaks["chr"]==chrom) & (tts_peaks["strand"]==strand)].sort_values("tts_pos")
    if tts_grp.empty: 
        continue

    tss_positions = grp["tss_pos"].to_numpy()
    tts_positions = tts_grp["tts_pos"].to_numpy()
    tts_scores    = tts_grp["tts_score"].to_numpy()

    if strand == "+":
        idx = np.searchsorted(tts_positions, tss_positions, side="right")  # first tts > tss
        has_downstream = idx < len(tts_positions)
        for i, ok in enumerate(has_downstream):
            if not ok: continue
            j = idx[i]
            dist = tts_positions[j] - tss_positions[i]
            if NOVEL_MIN_LEN <= dist <= NOVEL_MAX_LEN:
                pairs.append({
                    "chr": chrom, "strand": strand,
                    "tss_pos": int(tss_positions[i]),
                    "tss_score": float(grp.iloc[i]["tss_score"]),
                    "tts_pos": int(tts_positions[j]),
                    "tts_score": float(tts_scores[j]),
                    "length_bp": int(dist)
                })
    else:
        # On '-', downstream along transcription is toward decreasing coords
        idx = np.searchsorted(tts_positions, tss_positions, side="left") - 1  # last tts < tss
        has_downstream = idx >= 0
        for i, ok in enumerate(has_downstream):
            if not ok: continue
            j = idx[i]
            dist = tss_positions[i] - tts_positions[j]
            if NOVEL_MIN_LEN <= dist <= NOVEL_MAX_LEN:
                pairs.append({
                    "chr": chrom, "strand": strand,
                    "tss_pos": int(tss_positions[i]),
                    "tss_score": float(grp.iloc[i]["tss_score"]),
                    "tts_pos": int(tts_positions[j]),
                    "tts_score": float(tts_scores[j]),
                    "length_bp": int(dist)
                })

novel_pairs = pd.DataFrame(pairs)
print("Novel pairs:", len(novel_pairs))
if novel_pairs.empty:
    print(">>> Nothing paired. Likely causes:\n"
          "- MIN_PROB_FOR_PEAK too high (set to 0.6)\n"
          "- GENE_BUFFER too large (try 200–500)\n"
          "- NOVEL_MIN/MAX_LEN too strict (try 200..6000)\n"
          "- Very dense genes: few truly intergenic TSS survive")

# ===== 4) Optional: remove pairs overlapping any known gene ± buffer =====
def overlaps_any_gene(chrom, s, e, intervals, buf=500):
    if intervals is None or len(intervals)==0: return False
    ss = intervals[:,0] - buf
    ee = intervals[:,1] + buf
    return np.any((e >= ss) & (s <= ee))

keep = []
for (chrom, strand), grp in novel_pairs.groupby(["chr","strand"], sort=False):
    intervals = genes_by_chr.get(chrom, None)
    for r in grp.itertuples(index=False):
        a, b = (r.tss_pos, r.tts_pos) if strand=="+" else (r.tts_pos, r.tss_pos)
        s, e = (a if a<b else b), (b if b>a else a)
        if not overlaps_any_gene(chrom, s, e, intervals, buf=GENE_BUFFER):
            keep.append(r._asdict())

novel_final = pd.DataFrame(keep)
print("Novel pairs after gene-overlap removal:", len(novel_final))




In [ ]:

# Save
novel_final.to_csv(f"{out_dir}/novel_units.tsv", sep="\t", index=False)

novel_bed = pd.DataFrame({
    "chr": novel_final["chr"],
    "start": (np.minimum(novel_final["tss_pos"], novel_final["tts_pos"]) - 50).clip(lower=0),
    "end":   (np.maximum(novel_final["tss_pos"], novel_final["tts_pos"]) + 50),
    "name":  ("lncCand_" + novel_final.index.astype(str)),
    "score": ((novel_final["tss_score"] + novel_final["tts_score"]) * 500).round().astype(int),
    "strand": novel_final["strand"]
})
novel_bed.sort_values(["chr","start","end"]).to_csv(f"{out_dir}/novel_units.bed", sep="\t", header=False, index=False)

print("Wrote:",
      f"{out_dir}/protein_coding_TSS.tsv",
      f"{out_dir}/protein_coding_TTS.tsv",
      f"{out_dir}/novel_units.tsv",
      f"{out_dir}/novel_units.bed",
      sep="\n")

In [ ]:
#!/usr/bin/env python3
import re
import gzip
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ========= INPUTS =========
BED_PATH = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/prediction/novel_units.bed"
OUT_PNG  = "novel_units_genome_distribution.png"
OUT_PDF  = "novel_units_genome_distribution.pdf"

# Vis params
DOT_SIZE      = 18        # size of red dots
DOT_ALPHA     = 0.9       # opacity of dots
LINE_WIDTH    = 3.0       # chromosome line width
LEFT_MARGIN_BP= 0         # extra left padding in bp
RIGHT_MARGIN_BP=0         # extra right padding in bp
MAX_POINTS_PER_CHR = None # e.g., 3000 to subsample if too dense; None = plot all

# ========= HELPERS =========
def read_bed(path: str) -> pd.DataFrame:
    """Read a BED (>=3 cols). No header expected."""
    df = pd.read_csv(path, sep="\t", header=None, comment="#", engine="python")
    if df.shape[1] < 3:
        raise ValueError("BED must have at least 3 columns: chrom, start, end")
    cols = ["chrom","start","end","name","score","strand","rest"]
    df.columns = cols[:df.shape[1]]
    df["start"] = pd.to_numeric(df["start"], errors="coerce").astype("Int64")
    df["end"]   = pd.to_numeric(df["end"],   errors="coerce").astype("Int64")
    df = df.dropna(subset=["chrom","start","end"]).copy()
    return df

def natural_chr_key(s: str):
    """Sort Pf3D7_01_v3 ... Pf3D7_14_v3 numerically by the numeric token."""
    m = re.search(r"(\d+)", str(s))
    return (int(m.group(1)) if m else 1_000_000, str(s))

# ========= MAIN =========
def main():
    if not Path(BED_PATH).exists():
        raise FileNotFoundError(f"BED not found: {BED_PATH}")
    bed = read_bed(BED_PATH)

    # Chromosome lengths (infer from max end in BED)
    chr_len = bed.groupby("chrom", as_index=False)["end"].max()
    chr_len = chr_len.sort_values("chrom", key=lambda s: s.map(natural_chr_key))
    chroms  = chr_len["chrom"].tolist()
    sizes   = chr_len["end"].astype(int).tolist()

    # Compute dot positions: use feature midpoints by default
    bed["pos"] = ((bed["start"].astype(int) + bed["end"].astype(int)) // 2)

    # Build per-chromosome lists of dot x-coords
    dots_by_chr = {
        c: bed.loc[bed["chrom"] == c, "pos"].astype(int).to_numpy()
        for c in chroms
    }

    # Optional: subsample if too dense
    if MAX_POINTS_PER_CHR is not None:
        rng = np.random.default_rng(42)
        for c in chroms:
            arr = dots_by_chr[c]
            if arr.size > MAX_POINTS_PER_CHR:
                idx = rng.choice(arr.size, size=MAX_POINTS_PER_CHR, replace=False)
                dots_by_chr[c] = arr[idx]

    # ---- Plot ----
    n_chr = len(chroms)
    fig_h = max(6, 0.5 * n_chr)   # scale height with number of chromosomes
    plt.figure(figsize=(10, fig_h))

    # nice spaced y positions (top to bottom)
    ys = np.arange(n_chr)[::-1] + 1  # 1..n, top -> bottom
    ytick_labels = [c.replace("Pf3D7_", "Chr").replace("_v3","") for c in chroms]

    # global x range (0 .. max chr length) with margins
    xmax = int(max(sizes) + RIGHT_MARGIN_BP)
    xmin = int(0 - LEFT_MARGIN_BP)

    # draw chromosomes
    cmap = plt.get_cmap("tab20")
    for i, (c, size) in enumerate(zip(chroms, sizes)):
        y = ys[i]
        # chromosome line
        plt.hlines(y, xmin, size, color=cmap(i % 20), linewidth=LINE_WIDTH)
        # red dots
        x = dots_by_chr[c]
        if x.size:
            # keep dots within [xmin, size]
            x = x[(x >= xmin) & (x <= size)]
            plt.scatter(
                x, np.full_like(x, y, dtype=float),
                s=DOT_SIZE, facecolors="none", edgecolors="red", linewidths=1.5, alpha=DOT_ALPHA, zorder=3
            )

    plt.yticks(ys, ytick_labels)
    plt.ylim(ys.min()-1, ys.max()+1)
    plt.xlim(xmin, xmax)
    plt.xlabel("Genomic position (bp)")
    plt.title("Novel gene/lncRNA distribution across chromosomes")
    plt.grid(False)
    plt.tight_layout()
    plt.savefig(OUT_PNG, dpi=300)
    plt.savefig(OUT_PDF)
    plt.close()

    # quick log
    total = sum(len(v) for v in dots_by_chr.values())
    print(f"[done] Plotted {total} features on {n_chr} chromosomes.")
    print(f"[saved] {OUT_PNG}")
    print(f"[saved] {OUT_PDF}")

if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ==== Input ====
BED_PATH = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/prediction/novel_units.bed"
OUT_CURVE = "novel_units_length_curve_0_5000.png"

def read_bed(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", header=None, comment="#")
    if df.shape[1] < 3:
        raise ValueError("BED must have at least 3 columns")
    df = df.iloc[:, :3]
    df.columns = ["chrom", "start", "end"]
    df["start"] = pd.to_numeric(df["start"], errors="coerce").astype("Int64")
    df["end"]   = pd.to_numeric(df["end"],   errors="coerce").astype("Int64")
    df = df.dropna(subset=["start", "end"])
    df["length"] = (df["end"] - df["start"]).astype(int)
    return df

def main():
    df = read_bed(BED_PATH)

    # Clamp to 0–5000
    lengths = df["length"]
    lengths = lengths[(lengths >= 0) & (lengths <= 5000)]

    # Bin lengths (every 50 bp, can adjust)
    bins = np.arange(0, 5001, 50)
    counts, edges = np.histogram(lengths, bins=bins)

    # Midpoints for curve x-axis
    mids = (edges[:-1] + edges[1:]) / 2

    # Plot
    plt.figure(figsize=(8,6))
    plt.plot(mids, counts, color="blue", linewidth=2)
    plt.fill_between(mids, counts, step="mid", alpha=0.2, color="blue")
    plt.xlabel("Novel gene/lncRNA length (bp)")
    plt.ylabel("Count")
    plt.title("Novel gene/lncRNA length distribution")
    plt.xlim(0, 5000)
    plt.tight_layout()
    plt.savefig(OUT_CURVE, dpi=300)
    plt.show()

    print(f"[saved] {OUT_CURVE}")

if __name__ == "__main__":
    main()
